# Purpose

Calculate MGC Scores for GO terms

In [4]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt

### Load Brain Cell Type Profiles

As our first example lets calculate Brain MGC scores. These scores represent the GO terms' marker gene content with respect to brain cells.

In [3]:
path_brain_ctp = "/space/grp/aadrian/Pseudobulk_Function_Pipeline_HighRes/bin/bulkSimulationOneProfile/data/boot_run_feb29/1/CTProfiles/exp_brain_sc_with_metadata_pc_cpm_cell_type_profiles.csv"
brain_ctp = pd.read_csv(path_brain_ctp, index_col=0)
brain_ctp

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000286098,ENSG00000286106,ENSG00000286112,ENSG00000286190,ENSG00000286219,ENSG00000286237,ENSG00000287542,ENSG00000288602,ENSG00000288649,ENSG00000288675
Astrocytes,32.819733,0.000000,52.635128,21.118639,6.669843,0.000000,2.053207,20.644949,70.065130,11.393249,...,9.037270,19.770655,57.748180,31.514500,8.320486,52.138138,68.117615,38.330547,0.000000,5.076544
Excitatory neurons,1.170022,0.093254,48.352688,21.304518,13.754496,0.216956,0.857144,11.913785,30.244040,6.055134,...,2.241183,1.169713,42.195760,38.369793,4.543709,92.649925,100.205020,32.471050,0.012149,0.688493
Inhibitory neurons,1.227179,0.097602,42.944640,17.141390,13.448284,0.172491,1.483237,15.972332,41.043983,6.860017,...,0.761160,1.093804,46.746693,28.398600,3.828931,101.338160,139.156220,24.477087,0.001229,0.589905
Microglial cells,5.270259,0.000000,55.271400,28.644049,89.148360,20.702814,95.865870,22.505318,67.121254,34.856483,...,11.016589,6.026886,25.544413,26.931417,0.000000,63.942547,171.809880,31.028368,0.000000,0.000000
Oligodendrocyte precursor cells,13.893607,0.000000,30.717295,31.539900,25.565258,1.376305,0.000000,7.081689,45.368656,7.559312,...,9.388611,3.880854,21.299791,19.334665,2.805114,77.494700,88.728810,25.002390,0.000000,1.814200
Oligodendrocytes,0.559028,0.000000,46.318546,19.015804,19.004663,0.427411,0.251452,20.273474,119.014404,17.665918,...,4.943358,0.274065,22.988039,10.256967,0.133372,48.310028,154.988390,25.158625,0.441380,0.109666


# Calculate Gini Index across Genes

Gini index is a measure of entropy. We assume here that genes with high marker-like quality will have high indexes

In [5]:
def gini_index(values: np.ndarray) -> float:
    """
    Compute the Gini Index for an array of values.
    
    Parameters:
    values (np.ndarray): A 1D array of numerical values.
    
    Returns:
    float: The Gini Index (between 0 and 1).
    """
    values = np.sort(values)  # Sort values in ascending order
    n = len(values)
    cumulative = np.cumsum(values)
    numerator = np.sum((2 * np.arange(1, n + 1) - n - 1) * values)
    denominator = n * np.sum(values)
    return numerator / denominator if denominator != 0 else 0

def compute_ginis(df):
	df = df.T
	lo_ginis = []
	for i, gene_df in df.iterrows():
		gini = gini_index(gene_df.values)
		lo_ginis.append(gini)
	gini_df = pd.DataFrame(lo_ginis, index = df.index, columns=["gini"])
	return gini_df

gini = compute_ginis(brain_ctp)
gini.to_csv("data/brain_gini.csv")

### Calculate MGC Scores for GO terms

Each GO term gets a MGC score based on the brain cell type profile-derrived gini indexes

In [8]:
# Load GO term annotations
go_annot_path = "/space/grp/aadrian/Pseudobulk_Function_Pipeline_HighRes/bin/preprocessing/preprocessGO_pipe/data/2024_march/data/processing/bp_annotations_withGeneData_qc_annotations.csv"
go_annot = pd.read_csv(go_annot_path)
go_annot.head()

,DB_Object_Symbol,GO ID,Aspect,DB Object Name,ensembl_gene_id,hgnc_symbol,uniprotswissprot,entrezgene_id,description
0,IRGM,GO:0000045,P,Immunity-related GTPase family M protein,ENSG00000237693,IRGM,A1A4Y4,345611.0,immunity related GTPase M [Source:HGNC Symbol;...
1,BECN2,GO:0000045,P,Beclin-2,ENSG00000196289,BECN2,A8MW95,441925.0,beclin 2 [Source:HGNC Symbol;Acc:HGNC:38606]
2,AP4M1,GO:0000045,P,AP-4 complex subunit mu-1,ENSG00000221838,AP4M1,NaN,9179.0,adaptor related protein complex 4 subunit mu 1...
3,ATG13,GO:0000045,P,Autophagy-related protein 13,ENSG00000175224,ATG13,NaN,9776.0,autophagy related 13 [Source:HGNC Symbol;Acc:H...
4,ULK1,GO:0000045,P,Serine/threonine-protein kinase ULK1,ENSG00000177169,ULK1,O75385,8408.0,unc-51 like autophagy activating kinase 1 [Sou...


### Summarize Gini index across GO terms

In [9]:
def mgc_all(go_annot: pd.DataFrame, gini: pd.DataFrame) -> pd.DataFrame:
    """Calculate MGES for all GO terms in go_annot, MGES for other genes, and perform Wilcoxon rank-sum test.

    Args:
        go_annot (pd.DataFrame): DataFrame containing info about what genes are annotated with GO terms.
        gini (pd.DataFrame): DataFrame with Gini index values; index = gene IDs.

    Returns:
        pd.DataFrame: DataFrame with GO terms as index and columns ['mgc_gini', 'mgc_gini_others', 'wilcoxon_stat', 'p_value'].
    """
    lo_go_terms = []
    lo_mgc = []
    lo_mgc_others = []
    lo_stat = []
    lo_p_value = []

    # Loop through each GO group
    for go, go_df in go_annot.groupby("GO ID"):
        # Genes in the GO term
        go_gene_ids = set(go_df['ensembl_gene_id'].values)
        
        # Gini scores for genes in the GO term
        go_gini_values = gini.loc[gini.index.isin(go_gene_ids)].values.flatten()

        # Gini scores for genes NOT in the GO term
        other_gini_values = gini.loc[~gini.index.isin(go_gene_ids)].values.flatten()

        # Calculate the MGES metrics
        mgc = np.median(go_gini_values) if len(go_gini_values) > 0 else np.nan
        mgc_other = np.median(other_gini_values) if len(other_gini_values) > 0 else np.nan

        # Perform Wilcoxon rank-sum test (Mann-Whitney U)
        if len(go_gini_values) > 0 and len(other_gini_values) > 0:
            stat, p_value = ranksums(go_gini_values, other_gini_values)
        else:
            stat, p_value = np.nan, np.nan

        # Collect results
        lo_go_terms.append(go)
        lo_mgc.append(mgc)
        lo_mgc_others.append(mgc_other)
        lo_stat.append(stat)
        lo_p_value.append(p_value)

    # Compile results into a DataFrame
    mgc_df = pd.DataFrame({
        'mgc_gini': lo_mgc,
        'mgc_gini_others': lo_mgc_others,
        'wilcoxon_stat': lo_stat,
        'p_value': lo_p_value
    }, index=lo_go_terms)

    return mgc_df


def mgc_one(go_df: pd.DataFrame, gini: pd.DataFrame) -> float:
    """Calculate the MGES for one GO term (median Gini of genes in the GO Term).

    Args:
        go_df (pd.DataFrame): DataFrame of genes in the GO term.
        gini (pd.DataFrame): DataFrame with Gini index values; index = gene IDs.

    Returns:
        float: MGES for the GO term.
    """
    lo_gene_ids = go_df['ensembl_gene_id'].values
    lo_gini_indexes = []

    for gene_id in lo_gene_ids:
        try:
            gini_index = gini.loc[gene_id].values[0]
            lo_gini_indexes.append(gini_index)
        except KeyError:
            continue

    if lo_gini_indexes:
        return np.median(lo_gini_indexes)
    else:
        return np.nan

def mgc_others(go_df: pd.DataFrame, gini: pd.DataFrame) -> float:
    """Calculate the MGES for genes NOT in the GO term.

    Args:
        go_df (pd.DataFrame): DataFrame of genes in the GO term.
        gini (pd.DataFrame): DataFrame with Gini index values; index = gene IDs.

    Returns:
        float: MGES for genes not in the GO term.
    """
    # Genes in the current GO term
    go_gene_ids = set(go_df['ensembl_gene_id'].values)
    
    # Filter out those genes from gini DataFrame
    other_genes_gini = gini.loc[~gini.index.isin(go_gene_ids)]

    if not other_genes_gini.empty:
        return np.median(other_genes_gini.values)
    else:
        return np.nan

# Usage
mgc_df = mgc_all(go_annot, gini)
mgc_df.to_csv("data/mgc_gini_brain.csv")
mgc_df

,mgc_gini,mgc_gini_others,wilcoxon_stat,p_value
GO:0000045,0.205555,0.359372,-4.297182,0.000017
GO:0000070,0.261048,0.358893,-2.042822,0.041070
GO:0000077,0.189123,0.359168,-3.807017,0.000141
GO:0000079,0.386707,0.358548,0.337209,0.735959
GO:0000082,0.261464,0.359118,-2.632904,0.008466
...,...,...,...,...
GO:2001237,0.326138,0.358756,-0.777849,0.436658
GO:2001238,0.402848,0.358615,0.770577,0.440958
GO:2001240,0.471196,0.358304,0.928503,0.353147
GO:2001243,0.322552,0.358867,-1.138993,0.254706


# Wrapper Function to perform MGC Calculations

Here we calculate MGC scores for PBMC profiles and all CT Profiles

In [11]:
def __main__(ct_profile_path, go_annot_path, save_path):
	"""Wrapper function for calculating MGC scores for a given set of ct profiles and GO annotation
	"""
	# load ctps
	ctp = pd.read_csv(ct_profile_path, index_col=0)
	# compuyte gini indexes
	gini = compute_ginis(ctp)
	# load go annotations
	go_annot = pd.read_csv(go_annot_path)
	# compuate mgc
	mgc_df = mgc_all(go_annot, gini)
	# save
	mgc_df.to_csv(save_path)


In [ ]:
# Calculate MGCs for PBMC
__main__( "/space/grp/aadrian/Pseudobulk_Function_Pipeline_HighRes/bin/bulkSimulationOneProfile/data/boot_run_feb29/1/CTProfiles/exp_pbmc_sc_with_metadata_pc_cpm_cell_type_profiles.csv",
			go_annot_path,
			save_path= 'data/mgc_gini_pbmc.csv')

In [12]:


# Calculate MGCs for all profiles
__main__( "/space/grp/aadrian/Pseudobulk_Function_Pipeline_HighRes/bin/simpleCTProfileEGAD/data/dev/0709/all_human_CT_profiles.csv",
			go_annot_path,
			save_path= 'data/mgc_gini_all.csv')
